# 3. Time — subtract the cycle, keep the residual

**The question this notebook answers: what is left, once the calendar is accounted for?**

A daily and weekly cycle sits underneath almost every message timestamp: people are awake
in the day and asleep at night, at their keyboard on weekdays and elsewhere at weekends.
Finding that cycle is not the finding — *"there is a daily rhythm"* is not news, and
nobody needed a dataset to learn it. It is the **boring, expected part**, and its only job
is to be subtracted so whatever is left over becomes legible.

Two showcases here, both making the same move:

1. **Flights** — raw vs rolling vs over-smoothed, and the straight-line lie a gap in the
   index tells you. Small, clean, checkable by hand.
2. **The Ubuntu IRC channels** — model the daily cycle, subtract it, and ask what a release
   day looks like once the *ordinary* shape of the day is accounted for. That is the payoff:
   an effect invisible in the raw series and invisible in the cycle itself, that only shows
   up once you normalise the level away and compare shapes.


In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import (
    CountValues,
    Filter,
    FlagDates,
    GroupAgg,
    MapValues,
    Pipeline,
    RollingAvg,
    Share,
    SortValues,
    SubtractBaseline,
    TimeFeatures,
    TransformBase,
)
from goad_toolkit.visualizer import (
    BarPlot,
    FacetPlot,
    GroupedBarPlot,
    HistogramPlot,
    HorizontalLine,
    LinePlot,
    PlotSettings,
)

from wa_analyzer.data import load_own_chat, load_showcase

## 3.1 The showcase: flights

Monthly airline passengers, 1949–1960. A trend, a yearly cycle that grows *with* the trend
(multiplicative, not additive — the swings get bigger as the level rises), and no missing
months. Small enough to reason about by eye.


In [ ]:
flights = load_showcase("flights")
flights["date"] = pd.to_datetime(flights["year"].astype(str) + "-" + flights["month"], format="%Y-%b")
flights = flights.set_index("date").sort_index()[["passengers"]]
flights.head()


In [ ]:
settings = PlotSettings(figsize=(10, 4), title="Monthly passengers, 1949-1960")
fig, ax = LinePlot(settings).plot(data=flights.reset_index(), x="date", y="passengers")

### Raw, smoothed, and over-smoothed — always side by side

A rolling window wide enough to look clean is often wide enough to erase the thing you were
looking for. There is no window that is simply "correct" — a 3-month window barely touches
the noise, a 12-month window (one full cycle) removes the *seasonality itself*, not just the
noise. Plot raw and smoothed together, always, so a reader can see what the window did.


In [ ]:
short_window = Pipeline().add(RollingAvg, column="passengers", window=3, rename=True)
long_window = Pipeline().add(RollingAvg, column="passengers", window=12, rename=True)

reasonable = short_window.apply(flights.reset_index())
over_smoothed = long_window.apply(flights.reset_index())

smoothing = PlotSettings(
    figsize=(13, 4),
    title="One series, two windows",
    subplot_titles=["A reasonable window", "Over-smoothed: the window IS the cycle"],
)

host = LinePlot(smoothing)
fig, axes = host.create_figure(n_plots=2)

for ax, frame, label in [(axes[0], reasonable, "3-month"),
                         (axes[1], over_smoothed, "12-month")]:
    host.plot_on_axes(LinePlot(smoothing), ax, data=frame, x="date", y="passengers",
                      color=smoothing.base_color, label="raw")
    host.plot_on_axes(LinePlot(smoothing), ax, data=frame, x="date",
                      y="passengers_rolling_avg", color=smoothing.highlight_color,
                      label=label)

The 12-month window does not just denoise the yearly cycle — it **is** one full cycle, so
averaging over it removes the seasonality itself, not the noise around it. If your claim is
about the yearly pattern, this window has quietly deleted your evidence.


### A gap in the index is not the same as a gap in the plot

Punch a hole in the middle of the series — drop four months, as if that stretch of data
never arrived — and plot both ways: naively, and after reindexing to the full monthly range.


In [ ]:
holey = flights.drop(flights.index[60:64])  # 1954, four months deliberately missing

full_index = pd.date_range(flights.index.min(), flights.index.max(), freq="MS")
reindexed = holey.reindex(full_index)

gaps = PlotSettings(
    figsize=(13, 4),
    title="Four missing months, plotted two ways",
    subplot_titles=["Plotted naively: the gap is invisible",
                    "Reindexed to the full range: the gap is a gap"],
)
host = LinePlot(gaps)
fig, axes = host.create_figure(n_plots=2)

# ax.plot on purpose, not LinePlot: seaborn drops missing rows before drawing, which
# would quietly reconnect the very gap the right panel exists to show.
axes[0].plot(holey.index, holey["passengers"], marker="o", markersize=3)
axes[1].plot(reindexed.index, reindexed["passengers"], marker="o", markersize=3)

The left panel draws a confident diagonal straight through four months that do not exist —
continuity is a gestalt principle, and here it lies for you, cheaply. `pd.date_range` plus
`.reindex()` makes the missing rows exist again, as `NaN`, and matplotlib breaks the line at
a `NaN` rather than connecting through it. An absence looks like an absence.

This is also the one figure in the lesson drawn with `ax.plot` instead of `LinePlot`, and
the reason is the lesson itself: seaborn drops rows with missing values before drawing, so
it would reconnect the gap the right panel exists to show. Know what your plotting layer
does with missing data before you trust it to show you missing data.

Whether to then fill those `NaN`s (with `0`, an interpolation, a rolling estimate) is a
separate decision, made after you have seen the honest gap — not before.

## 3.2 Loading the IRC showcase

Same corpus as lessons 1 and 2, same parse — imported this time, not rebuilt.
`build_irc_pipeline()` already produces `day_name` (lesson 1's `TimeFeatures` step); the
regex features it also adds (`has_url`, `n_question`, `addressed_to`) go unused here, same as
in lesson 2. This notebook needs two more columns the base pipeline does not make: a full
`timestamp` (the parse keeps `date` and the `hh`/`mm` integers separate) and an `hour`.

Both are pipeline steps, so they are **added to the imported pipeline** rather than patched
onto the frame afterwards: `BuildTimestamp` — living in `scripts/pipelines.py`, next to the
parse it completes — glues `date + hh:mm` together, and `TimeFeatures`, the same transform
lesson 1 used, derives `hour` from it. Extending someone else's pipeline with two
`.add(...)` calls is the point of pipelines being objects, and exactly what 01.3's your-turn
had you doing to your own.

In [ ]:
from scripts.pipelines import BuildTimestamp, build_irc_pipeline

pipeline = build_irc_pipeline()
pipeline.add(BuildTimestamp)
pipeline.add(TimeFeatures, name="clock", column="timestamp", features=["hour"])

msgs = pipeline.apply(load_showcase("ubuntu_irc"))
print(f"{len(msgs):,} messages across {msgs['channel'].nunique()} channels")
msgs.head()

### The unsurprising cycle

`FacetPlot` is the tool for "the same plot, once per category" — the small-multiples loop
from 2.1.2, packaged as one call: one panel per weekday, every panel drawn by the same
`HistogramPlot`, so the daily rhythm and how it differs Monday-to-Sunday are both visible
at once, without seven separate cells. `sharey=True` in the settings is what makes the
panels comparable: left to autoscale on their own, a quiet Sunday would look exactly as
busy as a loud Tuesday.

In [ ]:
uk = msgs[msgs["channel"] == "#ubuntu-uk"]
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

weekdays = PlotSettings(
    figsize=(12, 6),
    title="#ubuntu-uk: message volume by hour, one panel per weekday",
    xlabel="hour",
    max_cols=4,
    sharey=True,
)
fig, axes = FacetPlot(weekdays).plot(
    inner=HistogramPlot(weekdays), data=uk, by="day_name", order=order,
    x="hour", bins=range(25), stat="count", color="steelblue",
)

That is the cycle, and it says the two unsurprising things a cycle says: people sleep at
night, and weekends look different (more on that in 3.4). Neither is the finding — it is the
shape a specific day has to be compared *against* before whatever is unusual about it can
show through.

## 3.3 A global event overrides local time

*Does an Ubuntu release change the shape of the day, or only its height?*

`#ubuntu-uk` and `#ubuntu-it` are regional channels: different countries, different normal
working hours, different normal peak hour. An Ubuntu release is a global event — the same
moment, everywhere. Does it just make both channels busier (louder, same shape), or does it
change *when* people are there?

Ten releases fall inside this corpus's date range (2013–2017), and — worth noting before
anything else — **every one of them is a Thursday**. Ubuntu has released on Thursdays
throughout this window. That matters: comparing a release day against "all other days"
would silently compare a Thursday against a baseline that includes weekends, and lesson
3.4 is about to show `#ubuntu-uk` has a large weekday/weekend difference of its own. The fair
comparison is release Thursdays against *ordinary* Thursdays.


In [ ]:
release_dates = pd.to_datetime([
    "2013-04-25", "2013-10-17",  # 13.04, 13.10
    "2014-04-17", "2014-10-23",  # 14.04, 14.10
    "2015-04-23", "2015-10-22",  # 15.04, 15.10
    "2016-04-21", "2016-10-13",  # 16.04, 16.10
    "2017-04-13", "2017-10-19",  # 17.04, 17.10
])
release_weekdays = release_dates.day_name().unique().tolist()  # ty: ignore[unresolved-attribute]
print(f"release weekdays in this window: {release_weekdays}")


### Ordinary rhythms, pooled — a large, stable sample

With ~250 ordinary Thursdays per channel, pooling every message into one hourly total is
safe: no single day can dominate a quarter-of-a-thousand-day sample.

Every data decision on the way to the plot is a pipeline step with a name. `FlagDates`
marks the ten release days — the event-study workhorse: flag the dates once, then filter
or group on the flag. The `#ubuntu-uk` shares are then three steps: `Filter` to ordinary
Thursdays (`day_name` is already on the frame, from lesson 1's `TimeFeatures`),
`CountValues(normalize=True)` for the hourly shares, `SortValues` so the hours plot in
clock order. The `#ubuntu-it` corpus arrives as hourly totals rather than raw messages, so
its pipeline differs exactly in the middle: `GroupAgg` to sum the counts per hour, `Share`
to normalise them — the same decisions, stated as steps, with the one genuine difference
between the two datasets visible as the one step that changed.

In [ ]:
uk = (
    Pipeline()
    .add(FlagDates, column="date", dates=release_dates, feature="is_release")
    .apply(uk)
)

uk_share = (
    Pipeline()
    .add(Filter, expr="day_name == 'Thursday' and not is_release")
    .add(CountValues, column="hour", feature="share", normalize=True)
    .add(SortValues, column="hour")
    .apply(uk)
)

release_hourly = load_showcase("ubuntu_irc_release_hourly")
it_share = (
    Pipeline()
    .add(Filter, expr="channel == '#ubuntu-it' and not is_release")
    .add(GroupAgg, by="hour", column="messages", agg="sum")
    .add(Share, column="messages", feature="share")
    .apply(release_hourly)
)

rhythms = PlotSettings(
    figsize=(13, 4),
    title="Ordinary Thursdays, two channels",
    subplot_titles=[
        f"#ubuntu-uk, peak hour: {uk_share.loc[uk_share.share.idxmax(), 'hour']}",
        f"#ubuntu-it, peak hour: {it_share.loc[it_share.share.idxmax(), 'hour']}",
    ],
    xlabel="hour (UTC)",
    ylabel="share of messages",
)

host = BarPlot(rhythms)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(BarPlot(rhythms), axes[0], data=uk_share, x="hour", y="share",
                  color="steelblue")
host.plot_on_axes(BarPlot(rhythms), axes[1], data=it_share, x="hour", y="share",
                  color="darkorange")

Two channels, two different normal rhythms — `#ubuntu-uk` peaks mid-morning UK time,
`#ubuntu-it` peaks in the afternoon/evening. Expected: different countries, different days.

### Release days — a 9-to-10-day sample, where pooling lies

Only nine or ten release Thursdays exist per channel. Pool their raw counts and a single
unusually busy release can dominate the total — the same "smoothing away the finding"
problem as the rolling window in 3.1, one level up. Check first, rather than trust the
pool. The reshape the check needs — one row per day, one column per hour — is
`DayHourTable`, a pipeline step from `scripts/profiles.py`, and the same table feeds
everything below:

In [ ]:
from scripts.profiles import DayHourTable

release_days = (
    Pipeline()
    .add(Filter, expr="channel == '#ubuntu-it' and is_release")
    .add(DayHourTable, values="messages")
    .apply(release_hourly)
)

pd.DataFrame({
    "peak_hour": release_days.idxmax(axis=1),
    "total_messages": release_days.sum(axis=1),
}).sort_values("total_messages", ascending=False)

The busiest release day by far (2013-04-25, Ubuntu 13.04) peaks at a different hour than
most of the others, and at nearly triple the volume of a typical release day — pooled raw
counts, it would single-handedly set the "release day peak". The fix: normalise each day to
its own hourly *share* first, so a quiet release day and a loud one count equally, then
average the shares. Every release counts once, not once per message.

That fix is the **model**: an ordinary Thursday's own hourly shape, averaged the same way.
The **residual** is release day minus that model, hour by hour — not two overlapping cycles
for a reader to subtract by eye, but the subtraction already done, plotted against a
horizontal zero line. Positive means "more than an ordinary Thursday, at this hour";
negative means less; the cycle itself has already been divided out.

As code, the whole move is one pipeline per channel: `Filter` to the days the claim is
about, `DayHourTable` to reshape, then the model — `MeanOfDayShares`, a transform defined
*in this notebook* rather than imported, because it is the thing to be able to defend —
and finally `SubtractBaseline`, which merges the ordinary-Thursday profile in and leaves
the difference. Filter, reshape, model, subtract: the analysis reads as the four decisions
it is made of.

In [ ]:
class MeanOfDayShares(TransformBase):
    """Average each day's own hourly share, so no single day's volume dominates."""

    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        shares = data.div(data.sum(axis=1), axis=0)
        return shares.mean(axis=0).rename("share").rename_axis("hour").reset_index()


def profile_pipeline(expr: str, values: str | None = None) -> Pipeline:
    """Filter, reshape, model: the same three steps for all four profiles."""
    return (
        Pipeline()
        .add(Filter, expr=expr)
        .add(DayHourTable, values=values)
        .add(MeanOfDayShares)
    )


# The model: each channel's own ordinary-Thursday shape.
uk_ord_profile = profile_pipeline("day_name == 'Thursday' and not is_release").apply(uk)
it_ord_profile = profile_pipeline(
    "channel == '#ubuntu-it' and not is_release", values="messages"
).apply(release_hourly)

# The residual: the release-day pipeline ends by subtracting that model, so what
# flows out is the deviation, not the cycle plus the deviation.
uk_diff = (
    profile_pipeline("is_release")
    .add(SubtractBaseline, column="share", baseline=uk_ord_profile, on="hour",
         feature="difference")
    .apply(uk)
)
it_diff = (
    profile_pipeline("channel == '#ubuntu-it' and is_release", values="messages")
    .add(SubtractBaseline, column="share", baseline=it_ord_profile, on="hour",
         feature="difference")
    .apply(release_hourly)
)

residuals = pd.concat([
    uk_diff.assign(channel="#ubuntu-uk"),
    it_diff.assign(channel="#ubuntu-it"),
])

release_residual = PlotSettings(
    figsize=(9, 4),
    title="Release day minus an ordinary Thursday, by hour",
    xlabel="hour (UTC)",
    ylabel="share of that day's messages, release − ordinary",
)
lines = LinePlot(release_residual)
fig, ax = lines.plot(
    data=residuals, x="hour", y="difference", hue="channel",
    palette={"#ubuntu-uk": "steelblue", "#ubuntu-it": "darkorange"}, marker="o",
)
lines.plot_on(HorizontalLine(release_residual), y=0)

uk_peak = uk_diff.loc[uk_diff.difference.idxmax()]
it_peak = it_diff.loc[it_diff.difference.idxmax()]
print(f"#ubuntu-uk residual peak: {int(uk_peak.hour)}:00 UTC (+{uk_peak.difference:.1%})")
print(f"#ubuntu-it residual peak: {int(it_peak.hour)}:00 UTC (+{it_peak.difference:.1%})")

Both residuals turn positive at the same hour — **14:00 UTC** — on a day that otherwise
looks like noise around zero. `#ubuntu-it` spikes sharply right there (+7.9 points) before
dipping back down; `#ubuntu-uk` rises more gently and stays up through the afternoon
(14:00–19:00) before fading. `#ubuntu-it` also has a second, separate lift in the evening
(19:00–21:00) that `#ubuntu-uk` does not share — the two channels are not doing exactly the
same thing, only starting at the same clock time. That shared start is the finding: a global
event does not make everyone louder at their usual local hour, the cycle already divided out
says so directly; it adds activity at one shared moment, and what happens after that moment
still depends on the channel.

### The hypothesis this refutes: nobody stays up

The obvious guess — people stay up late for a release — predicts *more* night-hour activity
on release days. Check it, the same way: night share (01:00–06:00 UTC), ordinary Thursdays
vs release days, mean of per-day shares so the small release sample is not at the mercy of
one loud night. And put the answer on a plot rather than in a printout — four bars grouped
by channel is a comparison a reader can make in a glance; four printed percentages is
homework you left them.

In [ ]:
def night_share(day_hour_counts: pd.DataFrame) -> float:
    """Mean of each day's own 01:00-06:00 share -- the same per-day normalisation."""
    shares = day_hour_counts.div(day_hour_counts.sum(axis=1), axis=0)
    return shares.reindex(columns=range(1, 7), fill_value=0).sum(axis=1).mean()


def day_table(source: pd.DataFrame, expr: str, values: str | None = None) -> pd.DataFrame:
    return Pipeline().add(Filter, expr=expr).add(DayHourTable, values=values).apply(source)


night = pd.DataFrame([
    {"channel": "#ubuntu-uk", "day": "ordinary Thursday",
     "share": night_share(day_table(uk, "day_name == 'Thursday' and not is_release"))},
    {"channel": "#ubuntu-uk", "day": "release day",
     "share": night_share(day_table(uk, "is_release"))},
    {"channel": "#ubuntu-it", "day": "ordinary Thursday",
     "share": night_share(day_table(release_hourly,
                                    "channel == '#ubuntu-it' and not is_release",
                                    values="messages"))},
    {"channel": "#ubuntu-it", "day": "release day",
     "share": night_share(day_table(release_hourly,
                                    "channel == '#ubuntu-it' and is_release",
                                    values="messages"))},
])

nobody = PlotSettings(
    figsize=(8, 4),
    title="Nobody stays up: the night's share barely moves on release day",
    xlabel="",
    ylabel="share of the day's messages, 01:00-06:00 UTC",
    base_color="#cccccc",
    highlight_color="#c44e52",
)
GroupedBarPlot(nobody).plot(
    data=night, x="channel", y="share", hue="day",
    palette=[nobody.base_color, nobody.highlight_color],
)

The bars say it directly: neither channel's night share jumps on release day — if anything
`#ubuntu-uk`'s falls. Nobody stays up. What changes is *which* waking hours people are
active in, not whether they are awake at all.

### Verify before claiming a mechanism

It is tempting to say "14:00 UTC is when the release announcement email goes out, so that's
the mechanism." Check it, rather than assert it — the actual `Date:` header on the
`ubuntu-announce` mailing list, for the releases in this window:

| release | announcement time (UTC) |
| --- | --- |
| 13.04 | 12:07 |
| 14.04 | 17:09 |
| 16.04 | 16:17 |
| 17.04 | 13:09 |

That is a two-to-five-hour spread, not a fixed 14:00 — the mailing-list announcement does
not explain the convergence. The finding survives, but in a weaker, more honest form: **a
global event overrides local rhythm on release day**, at a time that is not simply "when the
email went out." What the actual trigger is (a blog post, social media, the download mirrors
going live, word of mouth) is not settled by this dataset, and the honest move is to say so
rather than invent a reason that sounds tidy.

## 3.4 The weekend dip is one channel, and the shape says why

The small multiples in 3.2 hinted at a weekday/weekend difference. Is it general, or
specific to one channel? A ratio is not a finding until you know what it is made of.

In [ ]:
nl = msgs[msgs["channel"] == "#ubuntu-nl"]

for name, channel_df in [("#ubuntu-uk", uk), ("#ubuntu-nl", nl)]:
    channel_df = channel_df.copy()
    channel_df["is_weekend"] = channel_df["date"].dt.dayofweek >= 5

    weekday_per_day = (~channel_df["is_weekend"]).sum() / channel_df.loc[~channel_df["is_weekend"], "date"].nunique()
    weekend_per_day = channel_df["is_weekend"].sum() / channel_df.loc[channel_df["is_weekend"], "date"].nunique()
    msg_ratio = weekend_per_day / weekday_per_day

    wd_authors = channel_df[~channel_df["is_weekend"]].groupby("date")["author"].nunique().mean()
    we_authors = channel_df[channel_df["is_weekend"]].groupby("date")["author"].nunique().mean()
    author_ratio = we_authors / wd_authors

    msg_per_author_ratio = (weekend_per_day / we_authors) / (weekday_per_day / wd_authors)

    print(f"{name}")
    print(f"  messages/day ratio (weekend:weekday) = {msg_ratio:.2f}")
    print(f"  distinct authors/day ratio           = {author_ratio:.2f}")
    print(f"  messages per author ratio             = {msg_per_author_ratio:.2f}")
    print()


The dip is not general — `#ubuntu-nl` barely moves. `#ubuntu-uk` roughly halves at the
weekend, and it is **more about who shows up than how much each person says**: distinct
authors fall further than messages-per-author.

But a ratio explains nothing on its own. The shapes do — and both channels get a panel,
because "the dip is specific to `#ubuntu-uk`" is a claim about a *comparison*, and a
comparison a reader cannot see is one they have to take on faith. One pipeline builds the
profile for either channel: `Filter` to the channel, `MapValues` to fold seven day names
into weekday/weekend, `GroupAgg` to count per (period, hour), and `Share(by="period")` so
each period's hours sum to one separately — a quiet weekend and a loud week become
comparable by shape, which is the entire point:

In [ ]:
def period_profile(channel: str) -> pd.DataFrame:
    """Hour shares per period, each period summing to one on its own."""
    return (
        Pipeline()
        .add(Filter, expr=f"channel == '{channel}'")
        .add(MapValues, column="day_name",
             mapping={"Saturday": "weekend", "Sunday": "weekend"},
             feature="period", default="weekday")
        .add(GroupAgg, by=["period", "hour"], agg="size")
        .add(Share, column="n", by="period", feature="share")
        .apply(msgs)
    )


weekend = PlotSettings(
    figsize=(13, 4),
    title="Only #ubuntu-uk trades the office for the evening",
    subplot_titles=["#ubuntu-uk", "#ubuntu-nl"],
    xlabel="hour (UTC)",
    ylabel="share of messages",
    sharey=True,
)
host = LinePlot(weekend)
fig, axes = host.create_figure(n_plots=2)
for ax, channel in zip(axes, ["#ubuntu-uk", "#ubuntu-nl"]):
    host.plot_on_axes(
        LinePlot(weekend), ax, data=period_profile(channel), x="hour", y="share",
        hue="period", palette={"weekday": "steelblue", "weekend": "crimson"},
        marker="o", markersize=3,
    )

Weekdays in `#ubuntu-uk` peak mid-morning UK time and decline through the day. Weekends
invert: quiet all morning, peaking in the evening. **The story: people chatting from
work.** The weekday peak sits at the start of UK office hours; at weekends the same
community turns up in the evening, from home.

`#ubuntu-nl` is the control that makes that story credible. Its weekday profile already
peaks in the evening — it was never an office-hours channel to begin with — and its
weekend line barely moves, because traffic that never depended on the working day has
nothing to lose when the working day goes away. Why two national channels developed
different habits, this dataset cannot say — a different community, different norms about
chatting from work, maybe just who its regulars happened to be — and the claim does not
need it to: the *contrast* carries the finding. The channel with the office-hours peak is
the channel with the office-sized dip.

What makes this credible rather than a coincidence dressed up as a story: the peak lands in
plausible working hours for the channel's own timezone, not a random hour; the traffic does
not vanish at the weekend, it *moves*; the direction was predictable before looking; and
the channel without the work-time shape shows no dip.

## 3.5 Your turn: the cumulative area chart

Same technique that showed the release-day and weekend shifts above — count, group, look at
the shape — pointed at your own chat. `plotly`'s area chart is worth having in the toolkit
for this: interactive, and cumulative counts make a growth story easy to read at a glance.


In [ ]:
own = load_own_chat()

In [ ]:
import plotly.express as px

own = own.copy()
own["date"] = pd.to_datetime(own["timestamp"]).dt.date

daily = own.groupby(["date", "author"]).size().reset_index(name="message_count")  # ty: ignore[no-matching-overload]
daily["cumulative"] = daily.groupby("author")["message_count"].cumsum()

fig = px.area(
    daily, x="date", y="cumulative", color="author", line_group="author",
    labels={"cumulative": "cumulative messages"},
    title="Cumulative messages per author, over time",
)
fig.show()

### Who's first — two authors, or a group

With exactly two authors, "who sends the first message of the day" is a clean +1/−1 signal
you can accumulate: a running balance that tells you whether one person has been
consistently opening the conversation. With more than two, that question stops being
binary — the group-chat alternative below reuses 3.2's small-multiples idea instead: each
author's own hourly rhythm, side by side.


In [ ]:
n_authors = own["author"].nunique()
print(f"{n_authors} authors in this chat")

In [ ]:
if n_authors == 2:
    daily_first = own.sort_values("timestamp").groupby("date")["author"].first()
    authors = sorted(daily_first.unique())
    balance = daily_first.map({authors[0]: 1, authors[1]: -1}).cumsum()  # ty: ignore[invalid-argument-type]

    first_mover = PlotSettings(
        figsize=(10, 4),
        title=f"Who sends the first message of the day?\n"
              f"+1 = {authors[0]}, -1 = {authors[1]}",
        ylabel="cumulative balance",
        xtick_rotation=45,
    )
    lines = LinePlot(first_mover)
    fig, ax = lines.plot(data=balance.rename("balance").reset_index(), x="date", y="balance")
    lines.plot_on(HorizontalLine(first_mover), y=0, linewidth=0.8)

In [ ]:
if n_authors > 2:
    pipeline = Pipeline()
    pipeline.add(TimeFeatures, column="timestamp", features=["hour"])
    own_enriched = pipeline.apply(own)

    rhythm = PlotSettings(
        figsize=(12, 6),
        title="Group chat: hourly rhythm per author",
        xlabel="hour",
        max_cols=4,
        sharey=True,
    )
    FacetPlot(rhythm).plot(
        inner=HistogramPlot(rhythm), data=own_enriched, by="author",
        x="hour", bins=range(25), stat="count", color="steelblue",
    )

## 3.6 Questions

1. **What is the boring cycle in your own chat, and what did you subtract it to see?**
   Name the cycle (daily, weekly) before naming what is left over — the order matters.
2. **Did a rolling window ever remove the thing you were checking for, not just the noise
   around it?** State the window you used and why that width, not a wider or narrower one.
3. **If you found a shift like the release-day one, did you check for a mundane explanation
   before a global one?** A shared event, a holiday, a change in who was even present.
4. **What claim would a reindexed gap have hidden from you, had you not reindexed?**
